## Objective 

This notebook performs paired statistical hypothesis testing to determine whether the proposed cross-attention multimodal model significantly outperforms multiple image-only backbone models.

The following baseline models are evaluated:

- CLIP
- ConvNeXt
- Swin
- ViT

For each comparison (cross_attention vs baseline), three paired statistical tests are conducted:

1. Paired t-test (parametric test)
2. Wilcoxon signed-rank test (non-parametric test)
3. Paired bootstrap test (resampling-based test)

All tests are one-sided and evaluate whether the cross-attention model performs better than the baseline model.

## Input Data

Paths are explicitly defined as:

Proposed model:
results/proposed/cross_attention/all_seeds_summary.csv

Baselines:
results/image_only/clip/all_seeds_summary.csv
results/image_only/convnext/all_seeds_summary.csv
results/image_only/swin/all_seeds_summary.csv
results/image_only/vit/all_seeds_summary.csv

## Output

All results will be saved to:

results/statistical_test/

Generated files:
- paired_t_test_results.csv
- wilcoxon_results.csv
- bootstrap_results.csv
- statistical_summary_table.csv

## Hypothesis

For each metric:

Null Hypothesis (H0):
The mean performance difference between cross_attention and baseline is less than or equal to zero.

Alternative Hypothesis (H1):
The cross_attention model performs better than the baseline.

All tests are paired and one-sided.

## Imports and Paths

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp, wilcoxon

# Explicit paths
PROPOSED_PATH = "results/proposed/cross_attention/all_seeds_summary.csv"

MODEL_PATHS = {
    "clip": "results/image_only/clip/all_seeds_summary.csv",
    "convnext": "results/image_only/convnext/all_seeds_summary.csv",
    "swin": "results/image_only/swin/all_seeds_summary.csv",
    "vit": "results/image_only/vit/all_seeds_summary.csv"
}

OUTPUT_DIR = "results/statistical_test_2_sided"
os.makedirs(OUTPUT_DIR, exist_ok=True)

METRICS = ["macro_f1", "accuracy"]
ALPHA = 0.05
BOOTSTRAP_ITERATIONS = 10000
RANDOM_SEED = 42

proposed_df = pd.read_csv(PROPOSED_PATH)

## Paired t-test

### Objective

To test whether the mean difference between the cross_attention model and a baseline model is significantly greater than zero.

### What is being tested?

For each seed:

difference = proposed_metric - baseline_metric

We test whether the average of these differences is greater than zero.

### Assumption

The paired differences follow an approximately normal distribution.

### Input

- Per-seed macro_f1 and accuracy
- Matched seeds across models

### Output

- Mean difference
- t-statistic
- p-value (one-sided)
- Decision to reject H0

In [2]:
# Two-sided paired t-test
# H0: mean(Fusion - Image-only) = 0
# H1: mean(Fusion - Image-only) != 0

t_test_results = []

for model_name, model_path in MODEL_PATHS.items():
    baseline_df = pd.read_csv(model_path)
    
    merged = proposed_df.merge(
        baseline_df,
        on="seed",
        suffixes=("_proposed", "_baseline")
    ).sort_values("seed")
    
    for metric in METRICS:
        # Paired difference: fusion model performance - image-only baseline performance
        d = merged[f"{metric}_proposed"] - merged[f"{metric}_baseline"]
        d = d.values
        
        # Two-sided paired t-test implemented as a one-sample t-test on paired differences
        result = ttest_1samp(
            d,
            popmean=0.0,
            alternative="two-sided"
        )
        
        t_test_results.append({
            "comparison": f"cross_attention vs {model_name}",
            "metric": metric,
            "n": len(d),
            "mean_difference": np.mean(d),
            "std_difference": np.std(d, ddof=1),
            "t_statistic": result.statistic,
            "p_value": result.pvalue,
            "test_type": "two-sided paired t-test",
            "reject_H0": result.pvalue < ALPHA
        })

t_test_df = pd.DataFrame(t_test_results)

t_test_df.to_csv(
    os.path.join(OUTPUT_DIR, "two_sided_paired_t_test_results.csv"),
    index=False
)

t_test_df

,comparison,metric,n,mean_difference,std_difference,t_statistic,p_value,test_type,reject_H0
0,cross_attention vs clip,macro_f1,10,-0.000389,0.009457,-0.130045,8.993909e-01,two-sided paired t-test,False
1,cross_attention vs clip,accuracy,10,-0.000505,0.009632,-0.165647,8.720958e-01,two-sided paired t-test,False
2,cross_attention vs convnext,macro_f1,10,0.067655,0.009232,23.173583,2.468311e-09,two-sided paired t-test,True
3,cross_attention vs convnext,accuracy,10,0.069728,0.009186,24.002639,1.807163e-09,two-sided paired t-test,True
4,cross_attention vs swin,macro_f1,10,0.054562,0.010047,17.172549,3.466853e-08,two-sided paired t-test,True
5,cross_attention vs swin,accuracy,10,0.054743,0.010495,16.493968,4.932955e-08,two-sided paired t-test,True
6,cross_attention vs vit,macro_f1,10,0.247781,0.049424,15.853746,6.968870e-08,two-sided paired t-test,True
7,cross_attention vs vit,accuracy,10,0.248385,0.049525,15.859928,6.945226e-08,two-sided paired t-test,True
